# 39-gate — does an EXPLICIT `--target_modules` reach the ViT→LLM connector?

Rung 39, the **blocking reachability gate**. Baseline = rung 21 arm `A2_lr`, run `21_lr_2e4_v1`,
**epoch 1**, `checkpoint-901`. The ONE variable: `--target_modules` gains the eight merger `Linear`
layers — **`all-linear` PLUS the eight names**, nine separate argv values. Everything else is A2,
verbatim.

## 🔴 THIS IS NOT THE RUNG

This notebook decides whether the rung exists in its declared form. It spends **minutes** of GPU
(5 optimiser steps on 32 rows, per leg) so that the 2.9 h arm is never spent on a flag that does
nothing. It is **not zero-GPU** — say so, it runs a real `swift sft`.

**Pre-registration: [`PLAN.md`](PLAN.md).** Read §3 (the landmine), §4 (these criteria) and §5 (the
branch this gate selects) before changing a single value here. A gate that fires is a FINDING, never
an obstacle (RULES §7) — if it fails, commit the CSV and stop. *"The connector is unreachable even
when named explicitly"* is a publishable result for about $0.50.


In [ ]:
# --- bootstrap -----------------------------------------------------------------
import json, logging, os, sys
from pathlib import Path

# Walk up to the repo root by a guaranteed marker, bounded so it can never run past /.
EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not ((REPO / ".git").exists() or (REPO / "src").is_dir()):
    REPO = REPO.parent
for p in (EXP / "_models", EXP / "_tools", REPO / "src"):
    if p.is_dir():
        sys.path.insert(0, str(p))

os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")

import reachability_gate as gate
print("repo:", REPO, "| exp:", EXP)
print("merger targets:", len(gate.MERGER_TARGETS), "| expected n_aligner:", gate.EXPECTED_N_ALIGNER)


## Sibling rungs (change ONE value, rerun, bump the tag)

- **39-gate-a:** `MAX_STEPS = 20` — if 5 steps ever proves too few for ms-swift to write a
  checkpoint. It does not change what is measured; the adapter's tensor census is set by the LoRA
  geometry, not by how long it trained.
- **39-gate-b:** `ROWS = 64` — same reasoning, and it costs another minute.

🔴 **NOT a sibling rung, ever:** dropping `all-linear` from `TARGET_MODULES`, joining the names into
one string, or relaxing a criterion in §4a of `PLAN.md`. The first two are different experiments; the
third is disabling a gate (RULES §7).


In [ ]:
# --- parameters (RAW LITERALS ONLY — papermill injects BELOW this cell) ----------
MAX_STEPS = 5                 # optimiser steps per leg
ROWS      = 32                # rows of A2's own corpus per leg
REPO_ROOT = "/workspace/repo_rodri"
POD_ID    = "y6h32tbhwhgxxe"
KERNEL    = "infer"
RUN_TAG   = "39_connector_v1"


In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) --
from dataclasses import replace

from pathlib import PurePosixPath   # the pod is Linux; a WindowsPath would mangle the argv

cfg = replace(
    gate.Config(),
    repo=PurePosixPath(REPO_ROOT),
    src=PurePosixPath(f"{REPO_ROOT}/experiments/18-count-aug/runs/18_count_aug_v1/train.jsonl"),
    out_root=EXP / "runs",
    max_steps=MAX_STEPS,
    rows=ROWS,
)

# 🔴 THE LANDMINE, defused and PROVEN defused before a single GPU second is spent.
# ms-swift `pipelines/train/tuner.py:93` returns EARLY on a *string* target_modules and silently
# ignores it: a joined value trains happily, exits rc=0 and adapts NOTHING. `assert_splat` RAISES
# unless the argv carries one `--target_modules` token followed by 9 separate non-flag values, none
# of which contains a separator. This is checkable with no GPU, so it is checked with no GPU.
argv = gate.build_argv(cfg, Path("preview.jsonl"))
vals = gate.assert_splat(argv, len(cfg.target_modules))
print(f"--target_modules carries {len(vals)} SEPARATE values:")
for v in vals:
    print("   ", v)
assert "--adapters" not in argv, "the gate must build the LoRA fresh (declared deviation #4)"
assert "--lora_dropout" in argv, "the gate mirrors the arm's recipe (declared deviation #5)"
print("\nfull argv:\n ", " ".join(argv))


In [ ]:
# --- BOTH legs, one notebook, one exit code -------------------------------------
# Subject leg (`--freeze_aligner false`) then control leg (`--freeze_aligner true`). Running them
# together is declared deviation #6: the chain needs a single papermill invocation and reads a
# single return code, and papermill's non-zero exit IS the blocking mechanism.
#
# ⚠️ NOT zero-GPU: two real `swift sft` runs of MAX_STEPS optimiser steps each, plus model load.
# Minutes, not seconds. `run_both` RAISES if either leg fails the pre-registered criteria.
result = gate.run_both(cfg)


In [ ]:
# --- the census, and the pre-registered criteria re-asserted here ---------------
# Re-asserted in the notebook as well as inside `run_both`: the criteria are the pre-registration
# and they must be legible in the artifact a reviewer opens, not only in a library.
import pandas as pd

subject, control = result["legs"]["subject"], result["legs"]["control"]
print(pd.DataFrame([
    {"leg": leg["leg"], "freeze_aligner": leg["freeze_aligner"],
     "total": leg.get("total_tensors"), "n_llm": leg.get("n_llm"), "n_vit": leg.get("n_vit"),
     "n_aligner": leg.get("n_aligner"), "n_orphans": leg.get("n_orphans"),
     "cfg_named_8": len(leg.get("adapter_config", {}).get("found", []))}
    for leg in (subject, control)
]).to_string(index=False))

print("\nA2's census for comparison: 720 = 504 llm + 216 vit + 0 aligner, 0 orphans")

assert subject["n_aligner"] > 0, "subject leg: the connector was NOT reached (PLAN.md §9 shape 1)"
assert subject["n_orphans"] == 0, "subject leg: orphan tensors are dropped from the optimiser"
assert subject["n_llm"] == gate.A2_N_LLM and subject["n_vit"] == gate.A2_N_VIT, \
    "coverage NOT preserved — the targets replaced all-linear instead of extending it (H2)"
assert subject["n_aligner"] == gate.EXPECTED_N_ALIGNER, \
    f"n_aligner={subject['n_aligner']}, sharp expectation {gate.EXPECTED_N_ALIGNER} — READ this"
assert not subject["adapter_config"]["missing"], "adapter_config.json does not name all 8 layers"

print(f"\nBRANCH: {result['branch']}  ->  the arm runs with --freeze_aligner "
      f"{str(result['arm_freeze_aligner']).lower()}")
print("Both branches were pre-declared in PLAN.md §5, before this gate ran.")


## Result

The gate wrote `RESULTS_reachability39.csv` and `RESULTS_reachability39.json` at the **experiment
root**, outside the gitignored `runs/`. Those two files are what the chain commits, and they are the
result whichever way it went:

- **PASS** → the arm is licensed, in the flag shape the `branch` field names. Read the numbers
  against `PLAN.md` §4a before launching: the sharp expectation is `n_aligner == 16`.
- **FAIL** → *the connector is unreachable even when named explicitly.* Publish it. It answers the
  open question left by `context/decisions/the-merger-is-unreachable-by-default.md` §4, and it closes
  the lane for about $0.50. Do **not** relax a criterion to get past it (RULES §7).

Next: `01_connector_arm.ipynb`, via the chain below.


In [ ]:
# --- render the chain into POD SCRATCH and print the line the HUMAN runs ---------
# A committed `.sh` is forbidden and a `.py` launcher is forbidden, so the chain is rendered text in
# /workspace/tmp — never a repo file. This cell writes it and PRINTS the command; the notebook does
# not shell out.
import chain

ccfg = chain.ChainConfig(repo_root=REPO_ROOT, pod_id=POD_ID, kernel=KERNEL, run_tag=RUN_TAG)
path = chain.write(ccfg)
print("rendered ->", path, "\n")
print(chain.launch_command(ccfg))
print("\n--- rendered chain ---\n")
print(chain.render(ccfg))
